# Adding a model: the integration surface

## Learning goals
- Enumerate what a new model must declare to vLLM-Omni
- Sketch a deploy config with stages and connector edges
- Understand the per-stage preprocess/transfer contract

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## What integration actually requires

Adding a model is mostly *declaration*, not new serving code: you describe its stages, wire the edges, and supply the per-stage preprocess/transfer functions. The runtime (scheduling, batching, KV management, connectors) is reused.

In [ ]:
# A minimal deploy config as a Python dict (real ones are YAML under vllm_omni/deploy/).
deploy = {
    'model': 'my-org/my-omni-model',
    'stages': [
        {'id': 0, 'role': 'thinker', 'kind': 'AR',    'output_connectors': ['to_stage_1']},
        {'id': 1, 'role': 'talker',  'kind': 'AR',    'input_connectors': ['from_stage_0'],
                                                        'output_connectors': ['to_stage_2']},
        {'id': 2, 'role': 'code2wav','kind': 'codec', 'input_connectors': ['from_stage_1']},
    ],
}
import json; print(json.dumps(deploy, indent=2))

## The preprocess/transfer contract

Each stage supplies a function that turns the previous stage's payload into its own input. The runner calls it every iteration for AR stages that fuse upstream output at each step. Below we validate a toy contract: the function must accept the upstream payload and return the keys the stage declares it needs.

In [ ]:
def make_transfer(required_keys):
    def transfer(payload):
        missing = [k for k in required_keys if k not in payload]
        if missing:
            raise KeyError(f'stage input missing {missing}')
        return {k: payload[k] for k in required_keys}
    return transfer
talker_input = make_transfer(['hidden', 'text'])
print(talker_input({'hidden': [1, 2, 3], 'text': 'hi', 'extra': 'ignored'}))

## Exercise

Call `talker_input` with a payload missing `hidden`. What error surfaces, and why is failing loudly at the edge better than passing a bad tensor downstream?

In [ ]:
# Solution
try:
    talker_input({'text': 'hi'})
except KeyError as e:
    print('caught at the edge:', e)
print('Failing here localizes the bug to the transfer contract, not a stage forward pass.')

## Source lab

Open `vllm_omni/deploy/qwen3_omni_moe.yaml` and one diffusion config. Map every field above to the real schema, and find where model code registers its stage classes.